# Feature selection evaluation (derived_8.3-feature-selection-1.0)

Evaluates selected feature sets on dataset `derived_8.3` using XGBoost 1.3-lite dual protocol (drift-weighted vs unweighted).

In [1]:
import importlib.util
import json
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score

PROJECT_ROOT = Path.cwd().resolve()
for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (p / "data" / "splits").is_dir() and (p / "Modeling").is_dir():
        PROJECT_ROOT = p
        break
sys.path.insert(0, str(PROJECT_ROOT))

EXP_DIR = PROJECT_ROOT / "notebooks" / "experiment" / "derived_8.3-feature-selection-1.0"
OUT_DIR = EXP_DIR / "artifacts" / "eval"
ARTIFACTS_DIR = EXP_DIR / "artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

try:
    dummy = xgb.XGBRegressor(n_estimators=1, device="cuda")
    dummy.fit(np.array([[1.0], [2.0]]), np.array([1.0, 2.0]))
    XGB_DEVICE = "cuda"
except Exception as e:
    XGB_DEVICE = "cpu"
    print(f"XGBoost CUDA probe failed ({e}); falling back to CPU.")

XGB_PARAMS_LITE = {
    "objective": "reg:squarederror",
    "max_depth": 8,
    "min_child_weight": 10,
    "reg_lambda": 1.5,
    "reg_alpha": 0.03,
    "subsample": 0.9,
    "colsample_bytree": 0.8,
    "n_estimators": 1500,
    "learning_rate": 0.01,
    "tree_method": "hist",
    "device": XGB_DEVICE,
    "n_jobs": -1,
    "random_state": SEED,
    "verbosity": 0,
}
BETA = 0.2
TARGET = "soil_moisture_5cm"

HAND_MDR_V25 = [
    "SMAP_sm_pm_interp_ema02", "V_rollmin_LST_modis_kobs30", "D_sin_DOY", "G_rain_sum_3d",
    "V_ema_G_API_kobs7", "V_rollmin_G_API_kobs30", "G_rain_sum_7d", "C_lag_LST_modis_kobs30",
    "C_lag_G_API_kobs1", "V_ema_G_API_kobs14", "V_rollmean_G_API_kobs14", "G_API", "G_DSLR",
    "SMAP_ampm_diff_interp", "V_rollmax_G_API_kobs30", "V_ema_G_API_kobs30", "V_rollmean_s2_b11_kobs7",
    "V_ema_LST_modis_kobs7", "V_rollmean_G_API_kobs7", "C_lag_s2_b11_kobs30", "A_d_E_SAR_diff_kobs14",
    "C_lag_LST_modis_kobs6", "A_d_LST_modis_kobs14", "A_d_SMAP_sm_interp_kobs14",
    "V_rollstd_SMAP_sm_interp_kobs30", "SMAP_sm_interp_grad7", "year_frac", "sin_year", "cos_year",
    "API_x_year", "SMAP_x_year", "slope", "elev", "K_slope_sin", "K_slope_cos", "K_aspect_cos",
    "J_clay_wfrac_b0", "J_sand_wfrac_b0",
]

print("Setup complete. XGB_DEVICE =", XGB_DEVICE)

Setup complete. XGB_DEVICE = cuda


## Helper functions

Define metric evaluation, temporal weighting, data loading, and model training utilities.

In [2]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    err = y_true - y_pred
    ae = np.abs(err)
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": float(np.sqrt(np.mean(err ** 2))),
        "ubRMSE": float(np.std(err)),
        "Bias": float(np.mean(err)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "Med|Err|": float(np.median(ae)),
        "Pearson": float(np.corrcoef(y_true, y_pred)[0, 1])
        if len(y_true) > 1 and np.std(y_pred) > 0
        else 0.0,
    }

def temporal_weights(dates, beta=BETA):
    years = pd.to_datetime(dates).dt.year.to_numpy().astype(float)
    t_max = years.max()
    w = np.exp(beta * (years - t_max))
    w = w / w.mean()
    return np.asarray(w).ravel()

def load_splits(dataset):
    base = PROJECT_ROOT / "data" / "splits" / dataset
    train = pd.read_csv(base / "train.csv", parse_dates=["date"])
    val = pd.read_csv(base / "val.csv", parse_dates=["date"])
    test = pd.read_csv(base / "test.csv", parse_dates=["date"])
    return train, val, test

def load_feature_sets_for_dataset(dataset):
    sets = {}
    sets["hand_mdr_v25"] = list(HAND_MDR_V25)
    v3_path = PROJECT_ROOT / "data/splits/derived_8.2/dataset_metadata.py"
    if v3_path.exists():
        spec = importlib.util.spec_from_file_location("dm82", v3_path)
        dm = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(dm)
        if hasattr(dm, "OVERALL_SELECTED_FEATURES_V3"):
            sets["V3_sota"] = list(dm.OVERALL_SELECTED_FEATURES_V3)
    v1_path = PROJECT_ROOT / "data/splits/derived_8.3/dataset_metadata.py"
    if v1_path.exists():
        spec3 = importlib.util.spec_from_file_location("dm83", v1_path)
        dm3 = importlib.util.module_from_spec(spec3)
        spec3.loader.exec_module(dm3)
        if hasattr(dm3, "OVERALL_SELECTED_FEATURES_V1"):
            sets["8.3_V1"] = list(dm3.OVERALL_SELECTED_FEATURES_V1)
    art = ARTIFACTS_DIR / dataset
    if art.is_dir():
        for path in sorted(art.glob("*/selected_features.json")):
            payload = json.loads(path.read_text())
            variant = payload.get("variant") or path.parent.name
            sets[f"v6_{variant}"] = list(payload["features"])
    return sets

def train_eval(train_df, val_df, test_df, features, weighted=True):
    tv = pd.concat([train_df, val_df], ignore_index=True)
    missing = [f for f in features if f not in tv.columns]
    if missing:
        raise ValueError(f"Missing features: {missing[:8]}...")

    X_tv = tv[features].apply(pd.to_numeric, errors="coerce")
    y_tv = pd.to_numeric(tv[TARGET], errors="coerce")
    X_te = test_df[features].apply(pd.to_numeric, errors="coerce")
    y_te = pd.to_numeric(test_df[TARGET], errors="coerce")

    tv_ok = y_tv.notna()
    te_ok = y_te.notna()
    X_tv, y_tv = X_tv.loc[tv_ok], y_tv.loc[tv_ok]
    X_te, y_te = X_te.loc[te_ok], y_te.loc[te_ok]
    dates_tv = tv.loc[tv_ok, "date"]

    w = temporal_weights(dates_tv, BETA) if weighted else None
    model = xgb.XGBRegressor(**XGB_PARAMS_LITE)
    model.fit(X_tv, y_tv, sample_weight=w)
    pred = np.asarray(model.predict(X_te)).ravel()
    metrics = compute_metrics(y_te, pred)

    years = pd.to_datetime(test_df.loc[te_ok, "date"]).dt.year.to_numpy()
    y_te_np = np.asarray(y_te, dtype=float).ravel()
    by_year = {}
    for yr in sorted(np.unique(years)):
        m = years == yr
        by_year[int(yr)] = compute_metrics(y_te_np[m], pred[m])
    return metrics, by_year, pred

## Run evaluation sweep on derived_8.3

Evaluates all feature sets under drift-weighted and unweighted settings.

In [3]:
all_rows = []
year_rows = []
train_df, val_df, test_df = load_splits("derived_8.3")
fsets = load_feature_sets_for_dataset("derived_8.3")

print(f"Loaded {len(fsets)} feature sets for derived_8.3")
for name, feats in fsets.items():
    for weighted in (True, False):
        label = "drift" if weighted else "no-drift"
        print(f"  training {name} (n={len(feats)}, {label}) ...", flush=True)
        metrics, by_year, _ = train_eval(train_df, val_df, test_df, feats, weighted=weighted)
        all_rows.append({
            "dataset": "derived_8.3",
            "feature_set": name,
            "n_features": len(feats),
            "weighted": weighted,
            **metrics,
        })

summary = pd.DataFrame(all_rows)
summary.to_csv(OUT_DIR / "metrics_summary.csv", index=False)

print("--- No drift (weighted=False) Leaderboard ---")
no_drift = summary[~summary.weighted].sort_values("R2", ascending=False)
print(no_drift[["feature_set", "n_features", "R2", "RMSE", "MAE", "Pearson"]].to_string(index=False))

print("--- With drift (weighted=True) Leaderboard ---")
drift = summary[summary.weighted].sort_values("R2", ascending=False)
print(drift[["feature_set", "n_features", "R2", "RMSE", "MAE", "Pearson"]].to_string(index=False))

Loaded 12 feature sets for derived_8.3
  training hand_mdr_v25 (n=38, drift) ...


  training hand_mdr_v25 (n=38, no-drift) ...


  training V3_sota (n=47, drift) ...


  training V3_sota (n=47, no-drift) ...


  training 8.3_V1 (n=55, drift) ...


...

  training v6_c4_hybrid (n=50, drift) ...


  training v6_c4_hybrid (n=50, no-drift) ...


  training v6_c5_rf (n=50, drift) ...


  training v6_c5_rf (n=50, no-drift) ...


--- No drift (weighted=False) Leaderboard ---
              feature_set  n_features       R2     RMSE      MAE  Pearson
                  V3_sota          47 0.633344 0.062925 0.046781 0.824779
             hand_mdr_v25          38 0.616917 0.064319 0.048399 0.819719
        v6_c2c_xgb_nocorr          55 0.616228 0.064377 0.046843 0.813849
                   8.3_V1          55 0.610878 0.064824 0.047158 0.812869
      v6_c2b_xgb_softcorr          55 0.610878 0.064824 0.047158 0.812869
  v6_c2d_xgb_softcorr_k65          65 0.605733 0.065252 0.047527 0.810600
 v6_c0_baseline_bypass_on          14 0.596939 0.065975 0.048340 0.797302
             v6_c4_hybrid          50 0.588769 0.066640 0.048515 0.802709
                 v6_c5_rf          50 0.588271 0.066681 0.048265 0.805813
                v6_c2_xgb          50 0.585817 0.066879 0.048538 0.802874
    v6_c3_xgb_no_coverage          50 0.585549 0.066901 0.048691 0.801697
v6_c1_baseline_bypass_off           9 0.534473 0.070904 0.051117 0